In [1]:
from helper_functions import LSE
from helper_functions import construct_R
from helper_functions import decomp_orthog

import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [2]:
N = 400
rng = np.random.default_rng(10)
inputs = rng.uniform(0.0,1.0,size = (N,6))
model_matrix = np.concatenate((inputs,inputs**2),axis = 1)
print(model_matrix)

a=2.0

[[0.95600171 0.20768181 0.82844489 ... 0.02228515 0.26296857 0.01847414]
 [0.68903648 0.84174772 0.425509   ... 0.91570738 0.68117441 0.1143896 ]
 [0.57576055 0.75330186 0.82710394 ... 0.87130738 0.02102346 0.55588985]
 ...
 [0.25311305 0.86399329 0.17489096 ... 0.54603371 0.28441293 0.80790971]
 [0.59824505 0.49599978 0.08263223 ... 0.87444858 0.86536076 0.11876718]
 [0.07155261 0.1064045  0.14876927 ... 0.15121008 0.34690563 0.04009062]]


In [3]:
def function_responses(model_input):

    resp_1 = np.dot(np.array([a,a,a,0,0,0,a/2,a/2,a/2,0,0,0]),model_input) + rng.normal(0,0.5)
    resp_2 = np.dot(np.array([-a,-a,-a,0,0,0,a/2,a/2,a/2,0,0,0]),model_input) + rng.normal(0,0.5)
    resp_3 = np.dot(np.array([0,0,0,a,a,a,0,0,0,-a/2,-a/2,-a/2]),model_input) + rng.normal(0,0.5)
    resp_4 = np.dot(np.array([0,0,0,-a,-a,-a,0,0,0,-a/2,-a/2,-a/2]),model_input) + rng.normal(0,0.5)

    return[resp_1,resp_2,resp_3,resp_4]

In [4]:
responses = np.zeros((4,N))
for n in range(N):
    four_resp = function_responses(model_matrix[n])
    responses[0,n] = four_resp[0]
    responses[1,n] = four_resp[1]
    responses[2,n] = four_resp[2]
    responses[3,n] = four_resp[3]
print(responses)

[[ 5.26292419  5.99167001  5.19246354 ...  3.27654663  2.2474694
   0.27713565]
 [-2.70464006 -2.49676651 -1.95412402 ... -2.1116361  -1.31477838
  -0.40248015]
 [ 1.13087848  2.8459222   3.10633671 ...  2.13054299  2.63161767
   1.89676147]
 [-2.0398836  -6.28565614 -5.36575861 ... -6.6393357  -5.95139452
  -2.53950938]]


In [5]:
LSE_1 = LSE(model_matrix,responses[0].T)
print(LSE_1)
LSE_2 = LSE(model_matrix,responses[1].T)
print(LSE_2)
LSE_3 = LSE(model_matrix,responses[2].T)
print(LSE_3)
LSE_4 = LSE(model_matrix,responses[3].T)
print(LSE_4)

[ 2.52102979  1.8956956   1.84093521 -0.4252728   0.23591859 -0.0192481
  0.54375107  1.15557387  1.0268808   0.23825997 -0.11739671 -0.04320648]
[-1.7900559  -1.91215459 -2.69800741 -0.35520049  0.4796743   0.4597898
  0.8731978   0.87407411  1.64355117  0.38995977 -0.46412747 -0.59681264]
[ 0.1288703   0.44595433 -0.70866537  2.10485266  1.36448692  1.88066449
 -0.04627678 -0.52557512  0.8634081  -0.91686426 -0.42996284 -0.80980227]
[-0.17746735  0.19868991  0.33253015 -2.38727145 -2.13493612 -2.06693207
  0.11876914 -0.20893076 -0.26410547 -0.65745367 -0.88776821 -0.8354456 ]


In [6]:
#START THE NON-NEGATIVE GARROTE ANALYSIS
#NEED TO CONSTRUCT R
R_1 = construct_R(model_matrix,responses[0].T)[0]
print(R_1)

R_2 = construct_R(model_matrix,responses[1].T)[0]
print(R_2)

R_3 = construct_R(model_matrix,responses[2].T)[0]
print(R_3)

R_4 = construct_R(model_matrix,responses[3].T)[0]
print(R_4)

[[ 2.41010879e+00  3.93701493e-01  1.52511336e+00 ...  5.30965969e-03
  -3.08716456e-02 -7.98202513e-04]
 [ 1.73708149e+00  1.59569745e+00  7.83334494e-01 ...  2.18176411e-01
  -7.99676345e-02 -4.94237190e-03]
 [ 1.45150950e+00  1.42803103e+00  1.52264476e+00 ...  2.07597669e-01
  -2.46808523e-03 -2.40180439e-02]
 ...
 [ 6.38105536e-01  1.63786827e+00  3.21962927e-01 ...  1.30097974e-01
  -3.33891425e-02 -3.49069351e-02]
 [ 1.50819360e+00  9.40264589e-01  1.52120581e-01 ...  2.08346091e-01
  -1.01590507e-01 -5.13151192e-03]
 [ 1.80386268e-01  2.01710549e-01  2.73874579e-01 ...  3.60273087e-02
  -4.07255800e-02 -1.73217455e-03]]
[[-1.7112965  -0.39711973 -2.23515044 ...  0.00869031 -0.12205094
  -0.0110256 ]
 [-1.23341382 -1.60955178 -1.14802643 ...  0.35708904 -0.31615175
  -0.06826916]
 [-1.03064357 -1.44042962 -2.23153255 ...  0.33977483 -0.00975757
  -0.33176209]
 ...
 [-0.45308651 -1.65208873 -0.47185711 ...  0.21293118 -0.13200385
  -0.48217073]
 [-1.07089209 -0.94842825 -0.222942

In [7]:
D_str_her = [[[],[],[],[],[],[],[0],[1],[2],[3],[4],[5]],
    [[],[],[],[],[],[],[0],[1],[2],[3],[4],[5]],
    [[],[],[],[],[],[],[0],[1],[2],[3],[4],[5]],
    [[],[],[],[],[],[],[0],[1],[2],[3],[4],[5]]]

In [8]:
orthog = decomp_orthog(responses,[R_1,R_2,R_3,R_4],D_str_her,2,t=300)
print(orthog)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-05-09
Set parameter TimeLimit to value 300
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i9-11900H @ 2.50GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 30 rows, 68 columns and 64 nonzeros
Model fingerprint: 0xe0faef87
Model has 312 quadratic objective terms
Model has 18 quadratic constraints
Variable types: 54 continuous, 14 integer (14 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [2e+01, 5e+03]
  QObjective range [1e-01, 2e+03]
  Bounds range     [1e+00, 4e+02]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]
Presolve added 26 rows and 0 columns
Presolve removed 0 rows and 4 columns
Presolve time: 0.00s
Pre

In [9]:
print(orthog[0][0]*LSE_1)
print(orthog[0][1]*LSE_2)
print(orthog[0][2]*LSE_3)
print(orthog[0][3]*LSE_4)

[ 2.46462898  1.86226947  1.82178575 -0.          0.         -0.
  0.5315862   1.13519805  1.01619916  0.         -0.         -0.        ]
[-1.71480524 -1.79475735 -2.55991132 -0.          0.          0.
  0.80103985  0.77191665  1.52275604  0.         -0.         -0.        ]
[ 0.          0.         -0.          2.12481705  1.40514428  1.90847839
 -0.         -0.          0.         -0.92556066 -0.44277436 -0.82035959]
[-0.          0.          0.         -2.35916389 -2.09999174 -2.04687095
  0.         -0.         -0.         -0.64971285 -0.87323732 -0.82733697]
